# Reducción de interferencias magnetohidrodinámicas en ECG adquiridos durante resonancia magnética

## 1. Carga y caracterización de los registros

Se utiliza la base de datos *Influence of the MHD effect on 12-lead and 3-lead ECGs recorded in 1T to 7T MRI scanners* (Krug Passand, 2021), publicada en PhysioNet (Goldberger et al., 2000). Contiene registros de ECG de sujetos sanos adquiridos dentro de resonadores de 1 T, 3 T y 7 T, sin secuencias de adquisición activas, por lo que la distorsión registrada corresponde principalmente al efecto magnetohidrodinámico (MHD). También incluye registros de referencia adquiridos fuera del resonador.

Los archivos están en formato **WFDB** (*WaveForm DataBase*), el formato estándar de PhysioNet para señales fisiológicas. Cada registro se compone de tres archivos:

- **`.hea` (encabezado):** archivo de texto con la frecuencia de muestreo, la cantidad de derivaciones, sus nombres, unidades y la duración del registro.
- **`.dat` (señal):** los valores muestreados de cada derivación, en formato binario.
- **`.qrs` (anotaciones):** la posición, en muestras, de cada complejo QRS, marcada manualmente por médicos o especialistas en ECG. Se utilizarán como referencia para evaluar la detección automática de latidos.

Para leerlos se utiliza la librería de Python **`wfdb`**, desarrollada por PhysioNet, que interpreta estos formatos y devuelve las señales como matrices de NumPy y las anotaciones como vectores de posiciones.

El nombre de cada registro codifica las condiciones de adquisición. Por ejemplo, `ECGMRI3T04Ff` corresponde a un campo de 3 T, sujeto 04 y posición Ff. Las posiciones son:

- **Hf** (*head first*): el sujeto ingresa al resonador con la cabeza primero.
- **Ff** (*feet first*): ingresa con los pies primero.
- **Pro** y **Sup**: decúbito prono y supino (solo en el resonador de 1 T, de campo vertical).
- **Out**: registro de referencia, fuera del resonador.

Algunos registros tienen un número final (por ejemplo, `ECGMRI3T09Ff1` a `Ff3`), que corresponde a mediciones repetidas en la misma posición.

En esta sección se leen los encabezados y las anotaciones de todos los registros, sin cargar todavía las señales completas, para construir una tabla descriptiva del conjunto.

In [7]:
import re                       # expresiones regulares, para interpretar el nombre de cada registro
from pathlib import Path        # manejo de rutas de archivos

import numpy as np              # operaciones numéricas con vectores y matrices
import pandas as pd             # tablas de datos
import matplotlib.pyplot as plt # gráficos
import wfdb                     # lectura de registros en formato WFDB (PhysioNet)

# Carpeta con el dataset. El notebook tiene que estar en la misma carpeta que "Data"
carpeta_data = Path("Data")

In [8]:
# El nombre de cada registro tiene la forma ECGMRI<campo>T<sujeto><posición><repetición>.
# La expresión regular separa esas partes:
#   (\d)            -> intensidad del campo en tesla (1, 3 o 7)
#   (\d{2})         -> número de sujeto (dos dígitos)
#   (Ff|Hf|Pro|Sup|Out) -> posición
#   (\d?)           -> número de repetición (opcional)
patron_nombre = re.compile(r"ECGMRI(\d)T(\d{2})(Ff|Hf|Pro|Sup|Out)(\d?)$")

filas = []

# Se recorre cada encabezado (.hea) de la carpeta, ordenados por nombre
for archivo_hea in sorted(carpeta_data.glob("*.hea")):
    nombre = archivo_hea.stem                  # nombre del registro, sin la extensión
    ruta = str(carpeta_data / nombre)          # wfdb recibe la ruta sin extensión

    partes = patron_nombre.match(nombre)
    if partes is None:
        raise ValueError(f"Nombre de registro no reconocido: {nombre}")
    campo, sujeto, posicion, repeticion = partes.groups()

    # rdheader lee solo el encabezado (no la señal), por lo que es rápido
    encabezado = wfdb.rdheader(ruta)

    # rdann lee las anotaciones manuales de QRS (archivo .qrs)
    anotaciones = wfdb.rdann(ruta, "qrs")

    duracion_s = encabezado.sig_len / encabezado.fs

    filas.append({
        "Registro": nombre,
        "Campo (T)": int(campo),
        "Sujeto": f"{campo}T{sujeto}",          # el sujeto se identifica junto con el campo
        "Posición": posicion,
        "Repetición": repeticion or "—",
        "Dentro del resonador": posicion != "Out",
        "Frecuencia (Hz)": encabezado.fs,
        "Derivaciones": encabezado.n_sig,
        "Nombres de derivaciones": ", ".join(encabezado.sig_name),
        "Unidades": ", ".join(sorted(set(encabezado.units))),
        "Duración (s)": round(duracion_s, 1),
        "QRS anotados": len(anotaciones.sample),
        # Frecuencia cardíaca media aproximada a partir de las anotaciones
        "FC media (lpm)": round(60 * len(anotaciones.sample) / duracion_s, 1),
    })

registros = pd.DataFrame(filas)

print(f"Registros leídos: {len(registros)}")
print(f"Duración total: {registros['Duración (s)'].sum() / 60:.1f} minutos")
display(registros)

Registros leídos: 53
Duración total: 226.6 minutos


,Registro,Campo (T),Sujeto,Posición,Repetición,Dentro del resonador,Frecuencia (Hz),Derivaciones,Nombres de derivaciones,Unidades,Duración (s),QRS anotados,FC media (lpm)
0,ECGMRI1T01Out,1,1T01,Out,—,False,1024,12,"I, II, III, aVR, aVL, aVF, V1, V2, V3, V4, V5, V6",mV,29.3,44,90.1
1,ECGMRI1T01Pro,1,1T01,Pro,—,True,1024,12,"I, II, III, aVR, aVL, aVF, V1, V2, V3, V4, V5, V6",mV,39.1,57,87.6
2,ECGMRI1T01Sup,1,1T01,Sup,—,True,1024,12,"I, II, III, aVR, aVL, aVF, V1, V2, V3, V4, V5, V6",mV,26.4,38,86.5
3,ECGMRI3T01Ff,3,3T01,Ff,—,True,1024,12,"I, II, III, aVR, aVL, aVF, V1, V2, V3, V4, V5, V6",mV,326.2,361,66.4
4,ECGMRI3T01Hf,3,3T01,Hf,—,True,1024,12,"I, II, III, aVR, aVL, aVF, V1, V2, V3, V4, V5, V6",mV,366.2,400,65.5
5,ECGMRI3T01Out,3,3T01,Out,—,False,1024,12,"I, II, III, aVR, aVL, aVF, V1, V2, V3, V4, V5, V6",mV,156.2,171,65.7
6,ECGMRI3T02Ff,3,3T02,Ff,—,True,1024,12,"I, II, III, aVR, aVL, aVF, V1, V2, V3, V4, V5, V6",mV,24.4,27,66.4
7,ECGMRI3T02Out,3,3T02,Out,—,False,1024,12,"I, II, III, aVR, aVL, aVF, V1, V2, V3, V4, V5, V6",mV,24.4,27,66.4
8,ECGMRI3T03Ff,3,3T03,Ff,—,True,1024,12,"I, II, III, aVR, aVL, aVF, V1, V2, V3, V4, V5, V6",mV,195.3,186,57.1
9,ECGMRI3T03Out,3,3T03,Out,—,False,1024,12,"I, II, III, aVR, aVL, aVF, V1, V2, V3, V4, V5, V6",mV,127.0,119,56.2


In [9]:
# Cantidad de registros según el campo magnético y la posición
print("Registros por campo y posición:")
display(pd.crosstab(registros["Campo (T)"], registros["Posición"], margins=True, margins_name="Total"))

# Cantidad de registros según la cantidad de derivaciones (equipo de 3 o de 12 derivaciones)
print("Registros por cantidad de derivaciones:")
display(registros["Derivaciones"].value_counts().rename("Registros"))

# Sujetos que tienen al menos un registro dentro del resonador y uno de
# referencia fuera. Son los que permiten comparar el ECG del mismo sujeto
# con y sin efecto MHD
con_referencia = (
    registros.groupby("Sujeto")["Dentro del resonador"]
    .agg(lambda x: x.any() and (~x).any())
)
sujetos_pareados = con_referencia[con_referencia].index.tolist()

print(f"Sujetos con registro dentro y fuera del resonador: {len(sujetos_pareados)}")
print(sujetos_pareados)

Registros por campo y posición:


Posición,Ff,Hf,Out,Pro,Sup,Total
Campo (T),,,,,,
1,0,0,1,1,1,3
3,25,6,4,0,0,35
7,5,5,5,0,0,15
Total,30,11,10,1,1,53


Registros por cantidad de derivaciones:


Derivaciones
12    39
3     14
Name: Registros, dtype: int64

Sujetos con registro dentro y fuera del resonador: 10
['1T01', '3T01', '3T02', '3T03', '3T04', '7T01', '7T02', '7T03', '7T04', '7T05']


### Control de registros duplicados

En la tabla descriptiva, los registros del sujeto 7T04 presentan exactamente la misma duración, cantidad de QRS anotados y frecuencia cardíaca que los del sujeto 3T01, en las tres posiciones. Lo mismo ocurre entre los registros 3T02Ff y 3T02Out. Como es muy poco probable que dos mediciones distintas coincidan en estos tres valores, se verifica si los archivos son copias.

Para eso se calcula una huella digital (hash SHA-256) del contenido de cada archivo de señal (`.dat`) y de anotaciones (`.qrs`). El hash es un código que se obtiene a partir de todos los bytes del archivo: dos archivos con el mismo contenido producen exactamente el mismo código, y cualquier diferencia, aunque sea de un solo valor, produce un código distinto. Así se puede comprobar si dos registros son idénticos sin comparar las señales muestra por muestra.

In [10]:
import hashlib  # cálculo de huellas digitales (hash) de archivos

# Nombres de los registros (sirve tanto si la tabla tiene el nombre como
# columna como si lo tiene como índice)
nombres = (
    list(registros["Registro"]) if "Registro" in registros.columns
    else list(registros.index)
)


def huella(archivo):
    """Hash SHA-256 de todos los bytes del archivo."""
    return hashlib.sha256(archivo.read_bytes()).hexdigest()


huellas_senal = [huella(carpeta_data / f"{n}.dat") for n in nombres]
huellas_qrs = [huella(carpeta_data / f"{n}.qrs") for n in nombres]

# Agrupar los registros que comparten la misma huella
for tipo, huellas in [("señal (.dat)", huellas_senal), ("anotaciones (.qrs)", huellas_qrs)]:
    grupos = pd.Series(nombres).groupby(huellas).apply(list)
    repetidos = grupos[grupos.apply(len) > 1]

    print(f"Grupos de registros con {tipo} idénticas: {len(repetidos)}")
    for lista in repetidos:
        print("   " + " = ".join(lista))
    print()

Grupos de registros con señal (.dat) idénticas: 1
   ECGMRI3T01Out = ECGMRI7T04Out

Grupos de registros con anotaciones (.qrs) idénticas: 3
   ECGMRI3T01Hf = ECGMRI7T04Hf
   ECGMRI3T01Ff = ECGMRI7T04Ff
   ECGMRI3T01Out = ECGMRI7T04Out



El registro `ECGMRI7T04Out` es una copia exacta de `ECGMRI3T01Out`. Los registros `ECGMRI7T04Ff` y `ECGMRI7T04Hf` tienen las mismas anotaciones que `ECGMRI3T01Ff` y `ECGMRI3T01Hf`, aunque sus archivos de señal no son idénticos. Como dos sesiones distintas no pueden presentar los latidos exactamente en las mismas posiciones, se comparan las señales de cada par para determinar si corresponden a la misma grabación.

Para cada par se calcula la **correlación de Pearson** entre las señales, derivación por derivación. Mide qué tan parecida es la forma de dos señales, independientemente de su escala: vale 1 si una es copia de la otra (aunque esté multiplicada por una constante) y valores cercanos a 0 si no se parecen. También se calcula la diferencia máxima entre ambas, en mV.

In [11]:
# Pares de registros a comparar
pares = [
    ("ECGMRI3T01Ff", "ECGMRI7T04Ff"),
    ("ECGMRI3T01Hf", "ECGMRI7T04Hf"),
    ("ECGMRI3T01Out", "ECGMRI7T04Out"),
    ("ECGMRI3T02Ff", "ECGMRI3T02Out"),
]

filas_pares = []
for nombre_a, nombre_b in pares:
    # rdrecord lee la señal completa; p_signal es la matriz (muestras x derivaciones) en mV
    a = wfdb.rdrecord(str(carpeta_data / nombre_a)).p_signal
    b = wfdb.rdrecord(str(carpeta_data / nombre_b)).p_signal

    # Si tienen distinta longitud, se comparan solo las muestras en común
    n = min(len(a), len(b))
    a, b = a[:n], b[:n]

    # Correlación de cada derivación entre ambos registros
    correlaciones = [np.corrcoef(a[:, i], b[:, i])[0, 1] for i in range(a.shape[1])]

    filas_pares.append({
        "Registro A": nombre_a,
        "Registro B": nombre_b,
        "Muestras comparadas": n,
        "Correlación mínima": min(correlaciones),
        "Correlación media": np.mean(correlaciones),
        "Diferencia máxima (mV)": np.nanmax(np.abs(a - b)),
    })

display(pd.DataFrame(filas_pares).round(4))

,Registro A,Registro B,Muestras comparadas,Correlación mínima,Correlación media,Diferencia máxima (mV)
0,ECGMRI3T01Ff,ECGMRI7T04Ff,334001,0.9352,0.9485,7.7055
1,ECGMRI3T01Hf,ECGMRI7T04Hf,375001,0.9219,0.9455,17.5597
2,ECGMRI3T01Out,ECGMRI7T04Out,160000,1.0000,1.0000,0.0000
3,ECGMRI3T02Ff,ECGMRI3T02Out,25000,-0.1159,-0.0079,3.7222


In [12]:
# Registros excluidos: los tres de 7T04 duplican la sesión de 3T01
# (7T04Out es una copia exacta y 7T04Ff/7T04Hf comparten las anotaciones
# y tienen correlación > 0,92 con 3T01Ff/3T01Hf)
excluidos = ["ECGMRI7T04Ff", "ECGMRI7T04Hf", "ECGMRI7T04Out"]

registros["Excluido"] = [n in excluidos for n in nombres]
registros_validos = registros[~registros["Excluido"]].copy()

dentro = registros_validos["Dentro del resonador"]
print(f"Registros válidos: {len(registros_validos)} "
      f"({dentro.sum()} dentro y {(~dentro).sum()} fuera del resonador)")

# Comprobación con la descripción de PhysioNet (43 registros dentro, 203 minutos),
# usando la tabla completa, antes de excluir
dentro_total = registros["Dentro del resonador"]
print(f"Duración de los registros dentro del resonador (tabla completa): "
      f"{registros.loc[dentro_total, 'Duración (s)'].sum() / 60:.1f} min "
      f"en {dentro_total.sum()} registros")

print("\nRegistros válidos por campo y posición:")
display(pd.crosstab(registros_validos["Campo (T)"], registros_validos["Posición"],
                    margins=True, margins_name="Total"))

con_referencia = (
    registros_validos.groupby("Sujeto")["Dentro del resonador"]
    .agg(lambda x: x.any() and (~x).any())
)
sujetos_pareados = con_referencia[con_referencia].index.tolist()
print(f"Sujetos con registro dentro y fuera del resonador: {len(sujetos_pareados)}")
print(sujetos_pareados)

Registros válidos: 50 (41 dentro y 9 fuera del resonador)
Duración de los registros dentro del resonador (tabla completa): 203.1 min en 43 registros

Registros válidos por campo y posición:


Posición,Ff,Hf,Out,Pro,Sup,Total
Campo (T),,,,,,
1,0,0,1,1,1,3
3,25,6,4,0,0,35
7,4,4,4,0,0,12
Total,29,10,9,1,1,50


Sujetos con registro dentro y fuera del resonador: 9
['1T01', '3T01', '3T02', '3T03', '3T04', '7T01', '7T02', '7T03', '7T05']


### Resultado de la caracterización de los registros

El conjunto contiene 53 registros, todos muestreados a 1024 Hz y con las señales expresadas en mV:

- 43 registros dentro del resonador, con una duración total de 203,1 minutos, que coincide con la descripción de PhysioNet, y 10 registros de referencia fuera del resonador.
- 39 registros de 12 derivaciones (I, II, III, aVR, aVL, aVF y V1 a V6), adquiridos con el Holter CardioMem 3000, y 14 de 3 derivaciones (I, II y III), adquiridos con el monitor inalámbrico del equipo de resonancia (Tesla M3). Los de 3 derivaciones corresponden a los sujetos 3T10 a 3T23, todos en posición Ff y sin registro de referencia.
- Los registros a 1 T son cortos (26 a 39 s), mientras que los de 3 T y 7 T duran entre 24 s y 12 minutos. La frecuencia cardíaca media, estimada a partir de las anotaciones, varía entre 46 y 90 lpm.

**Registros duplicados:** El control de huellas digitales mostró que `ECGMRI7T04Out` es una copia exacta de `ECGMRI3T01Out`, y que `ECGMRI7T04Ff` y `ECGMRI7T04Hf` comparten las anotaciones de `ECGMRI3T01Ff` y `ECGMRI3T01Hf`. Sus señales presentan una correlación de entre 0,92 y 0,95 con los latidos alineados muestra a muestra, lo que no es posible entre sesiones distintas (como referencia, los registros 3T02Ff y 3T02Out, que sí son sesiones distintas, tienen una correlación cercana a 0). Por lo tanto, corresponden a una misma sesión. Se excluyen los tres registros de 7T04 y se conservan los de 3T01, ya que la serie de 3 T es consecutiva (sujetos 01 a 23) y coincide con los 23 sujetos informados en el dataset. Como no es posible confirmar cuál de las dos etiquetas de campo es la correcta, esto se considera una limitación del trabajo.

Quedan 50 registros válidos y 9 sujetos con registro dentro y fuera del resonador, que se utilizarán para comparar la morfología del ECG con y sin efecto MHD. Los 50 registros, al tener anotaciones manuales de QRS, se utilizarán para evaluar la detección de latidos.